In [45]:
from langchain.tools import tool
# or this: from langchain_core.tools import tool
from langchain_ollama import ChatOllama

In [46]:
@tool
def multiply(a:int,  b : int) ->int:
    """Multiplies two numbers together, Use this for math operations"""
    return a * b  

In [47]:
@tool
def get_employee_email(name: str) -> str:
    """Get employee email by name, use this for Hr operations."""
    # In a real application, this function would query a database or an API to get the email address.
    # For demonstration purposes, we'll return a mock email address.
    #Mock databse
    db={"John Doe": "john.doe@company.com", "Jane Smith": "jane.smith@company.com"}


In [48]:
from pydantic import BaseModel , Field
from langchain_core.tools.structured import StructuredTool

In [49]:
class TicketInput(BaseModel):
    issue_type:str=Field(description="Type of the issue, e.g., 'software', 'hardware', 'network'")
    priority:int=Field(description="Priority of the issue(1 -5), e.g., 'low'-1, 'medium'-3, 'high'-5'")
    description:str=Field(description="Detailed description of the issue")

In [50]:
def create_it_ticket(issue_type: str, priority: int, description: str) -> str:
    """Creates an IT support ticket based on the provided details."""
    # In a real application, this function would interact with a ticketing system or database.
    # For demonstration purposes, we'll return a mock ticket ID.
    ticket_id = f"TICKET-{issue_type[:3].upper()}-{priority}-{len(description)}"
    return f"Ticket created successfully! Ticket ID: {ticket_id}, Issue Type: {issue_type}, Priority: {priority}, Description: {description}"

In [51]:
ticket_tool= StructuredTool.from_function(
    func=create_it_ticket,
    name="create_ticket",
    description="Creates an IT support ticket based on the provided details.",
    args_schema=TicketInput)

In [52]:
ticket_tool.invoke({"issue_type": "software", "priority": 3, "description": "Application is not responding."})

'Ticket created successfully! Ticket ID: TICKET-SOF-3-30, Issue Type: software, Priority: 3, Description: Application is not responding.'

In [53]:
ticket_tool.invoke({
    "issue_type": "software",
    "priority": 3,
    "description": "Application is not responding."
})

'Ticket created successfully! Ticket ID: TICKET-SOF-3-30, Issue Type: software, Priority: 3, Description: Application is not responding.'

In [54]:

tools=[multiply, get_employee_email, ticket_tool]

llm= ChatOllama(model="gpt-oss:120b-cloud")

llm_with_tools=llm.bind_tools(tools)

query= "My laptop screen is broken , It's urgent and multiply 15 and 17"

response= llm_with_tools.invoke(query)

print("----- RAW LLM Reponse -----:")
print(f"Content: {response.content}")
print(f"Tool Calls: {response.tool_calls}")


----- RAW LLM Reponse -----:
Content: 
Tool Calls: [{'name': 'multiply', 'args': {'a': 15, 'b': 17}, 'id': '78a5aa22-8fdf-4138-8065-9732dc270001', 'type': 'tool_call'}]


In [55]:
from langchain_core.messages import  HumanMessage, AIMessage, ToolMessage
messages=[ HumanMessage(content="Who is prime minister of india? , also multiply 15 by 17")]

ai_msg=llm_with_tools.invoke(messages)
print(ai_msg)
messages.append(ai_msg)
print(ai_msg.tool_calls)


content='' additional_kwargs={} response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-07-20T05:21:17.540175357Z', 'done': True, 'done_reason': 'stop', 'total_duration': 982967142, 'load_duration': None, 'prompt_eval_count': 269, 'prompt_eval_duration': None, 'eval_count': 64, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'} id='lc_run--019f7df8-a90a-7d12-adb1-7b077c180d44-0' tool_calls=[{'name': 'multiply', 'args': {'a': 15, 'b': 17}, 'id': '1e34a5ee-ef8b-44a1-a21c-bef769d349a4', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 269, 'output_tokens': 64, 'total_tokens': 333}
[{'name': 'multiply', 'args': {'a': 15, 'b': 17}, 'id': '1e34a5ee-ef8b-44a1-a21c-bef769d349a4', 'type': 'tool_call'}]


In [60]:
for tool_call in ai_msg.tool_calls:
    selected_tool={
        "multiply" :multiply,
        "get_employee_email": get_employee_email,
        "create_ticket" : ticket_tool
    } [tool_call['name']]
    tool_output= selected_tool.invoke(tool_call["args"])
    
    tool_msg= ToolMessage(tool_output, tool_call_id=tool_call["id"])
    messages.append(tool_msg)
    print(f"Executed {tool_call['name']} with output: {tool_output}")


Executed multiply with output: 255


In [62]:
final_response= llm_with_tools.invoke(messages)
print(f"final Response: {final_response.content}")

final Response: The current Prime Minister of India is **Narendra Modi** (in office since May 2014).

The product of 15 × 17 is **255**.
